# BMIN 5200: Week 4 in-class exercise
## BFS, DFS, greedy best-first, and A*: comparing search cost

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week04.ipynb)

**Time:** ~25 minutes · **Pairs with:** Uninformed and heuristic search (state space graphs, BFS/DFS, evaluation functions, greedy best-first, A*)

### Tasks
- Find a route for a patient from the emergency department to an operating room on a weighted hospital map
- Implement a single `search()` function in which only the frontier ordering varies, and use it to obtain five algorithms
- Count node expansions and compare path cost with search effort across all five algorithms
- Show that greedy best-first search returns a suboptimal route, and that A* loses its optimality guarantee with an inadmissible heuristic

### Background
Patient transport, bed assignment, OR scheduling, and differential-diagnosis workups can all be
formulated as search over a weighted graph. The algorithms differ mainly in the information used
to order the frontier: the cost incurred so far, an estimate of the remaining cost, or neither.
The final part of the exercise examines a heuristic that overestimates the remaining cost. It
reduces the number of expansions but returns a worse route, and the output gives no indication
of the error. Optimization tools used in health systems are subject to the same failure mode.

## Setup

All required packages are preinstalled in Colab. `heapq`, the standard-library priority queue,
is the only data structure the search implementation requires.

In [ ]:
import heapq
import math

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

print("Setup complete.")

## Part 1: The map and a generic search function

The map is a synthetic hospital floor plan. Each unit has an (x, y) position measured in minutes
of walking, and the traversal time of each corridor is its straight-line length multiplied by a
congestion factor. Most factors are 1.0, and a few corridors are slower. Note the corridor
`south_lobby -> operating_room_3`: it is the shortest edge on the map geometrically, but it uses
a freight elevator shared with materials management and takes 13.4 minutes. Much of the behavior
in this exercise follows from this edge.

The task is to move one patient from `emergency_dept` to `operating_room_3`.

In [ ]:
# Positions are in minutes of walking, so straight-line distance is directly comparable
# to traversal time. Synthetic, but the shape is taken from a real transport problem.
UNITS = {
    "emergency_dept":      (0.0, 0.0),
    "ed_imaging":          (1.5, 2.5),
    "radiology_reading":   (3.0, 3.0),
    "central_lab":         (2.5, 1.0),
    "service_corridor":    (4.0, 1.0),
    "south_lobby":         (8.0, 1.0),
    "medical_icu":         (6.0, 2.5),
    "north_corridor":      (2.0, -4.0),
    "elevator_tower_b":    (6.0, -4.0),
    "step_down_unit":      (7.5, -1.0),
    "pre_op_holding":      (9.0, -2.0),
    "operating_room_3":    (10.0, 0.0),
    "main_entrance_lobby": (-5.0, 1.5),
    "outpatient_clinic":   (-8.0, 2.5),
    "registration_desk":   (-6.5, 3.5),
    "loading_dock":        (-6.0, -2.5),
}

# (from, to, congestion factor). 1.0 means you walk it at straight-line speed.
CORRIDORS = [
    ("emergency_dept", "service_corridor", 1.0),
    ("emergency_dept", "north_corridor", 1.0),
    ("emergency_dept", "central_lab", 1.0),
    ("emergency_dept", "ed_imaging", 1.3),
    ("emergency_dept", "main_entrance_lobby", 1.0),
    ("main_entrance_lobby", "outpatient_clinic", 1.0),
    ("main_entrance_lobby", "registration_desk", 1.0),
    ("main_entrance_lobby", "loading_dock", 1.0),
    ("outpatient_clinic", "registration_desk", 1.0),
    ("ed_imaging", "radiology_reading", 1.0),
    ("radiology_reading", "medical_icu", 1.4),
    ("central_lab", "service_corridor", 1.0),
    ("service_corridor", "south_lobby", 1.2),
    ("service_corridor", "medical_icu", 1.0),
    ("medical_icu", "south_lobby", 1.0),
    ("south_lobby", "step_down_unit", 1.0),
    ("south_lobby", "operating_room_3", 6.0),      # shared freight elevator
    ("north_corridor", "elevator_tower_b", 1.0),
    ("elevator_tower_b", "pre_op_holding", 1.0),
    ("elevator_tower_b", "step_down_unit", 1.0),
    ("step_down_unit", "pre_op_holding", 1.0),
    ("pre_op_holding", "operating_room_3", 1.0),
]

START, GOAL = "emergency_dept", "operating_room_3"


def straight_line(unit, other):
    (x1, y1), (x2, y2) = UNITS[unit], UNITS[other]
    return math.hypot(x2 - x1, y2 - y1)


hospital = nx.Graph()
for unit, position in UNITS.items():
    hospital.add_node(unit, position=position)
for here, there, congestion in CORRIDORS:
    hospital.add_edge(here, there, minutes=round(straight_line(here, there) * congestion, 1))

slowest = sorted(hospital.edges(data=True), key=lambda edge: -edge[2]["minutes"])[:5]
print(f"{hospital.number_of_nodes()} units, {hospital.number_of_edges()} corridors\n")
print("the five slowest corridors:")
for here, there, data in slowest:
    print(f"   {here:22s} -> {there:22s} {data['minutes']:5.1f} min")

In [ ]:
SHORT_NAMES = {
    "emergency_dept": "ED", "ed_imaging": "ED imaging", "radiology_reading": "rad reading",
    "central_lab": "lab", "service_corridor": "service corr", "south_lobby": "S lobby",
    "medical_icu": "MICU", "north_corridor": "N corr", "elevator_tower_b": "elev B",
    "step_down_unit": "step-down", "pre_op_holding": "pre-op", "operating_room_3": "OR 3",
    "main_entrance_lobby": "main lobby", "outpatient_clinic": "outpt clinic",
    "registration_desk": "registration", "loading_dock": "dock",
}
positions = nx.get_node_attributes(hospital, "position")

plt.figure(figsize=(11, 6))
nx.draw_networkx(hospital, pos=positions, labels=SHORT_NAMES, node_size=900, font_size=8)
nx.draw_networkx_edge_labels(
    hospital, pos=positions, font_size=7,
    edge_labels={(a, b): data["minutes"] for a, b, data in hospital.edges(data=True)})
plt.title("Hospital transport map: minutes between units (synthetic)")
plt.axis("off")
plt.show()

The cell below contains the full search algorithm. It maintains a frontier of routes that have
been generated but not yet expanded, removes one route at a time, and adds its extensions to
neighboring units. The five algorithms differ only in `frontier_policy`, which returns the sort
key for a route. Ordering by depth gives breadth-first search, ordering by cost so far gives
uniform-cost search, and ordering by estimated remaining cost gives greedy best-first search.
Read through the function, then run breadth-first search.

In [ ]:
def search(graph, start, goal, frontier_policy):
    """Generic graph search. `frontier_policy(unit, minutes_so_far, depth, order)`
    returns the sort key that decides what comes off the frontier next."""
    order = 0                       # generation counter, used to break ties predictably
    frontier = [(frontier_policy(start, 0.0, 0, order), order, start, [start], 0.0)]
    expanded = []                   # the instrumentation: every unit we actually opened

    while frontier:
        _, _, unit, route, minutes = heapq.heappop(frontier)
        if unit in expanded:
            continue                # we already opened this unit by a route we preferred
        expanded.append(unit)

        if unit == goal:
            return {"route": route, "minutes": round(minutes, 1), "stops": len(route) - 1,
                    "expanded": len(expanded), "expansion_order": expanded}

        for neighbor in sorted(graph.neighbors(unit)):
            if neighbor in expanded:
                continue
            order += 1
            step = graph[unit][neighbor]["minutes"]
            heapq.heappush(frontier,
                           (frontier_policy(neighbor, minutes + step, len(route), order),
                            order, neighbor, route + [neighbor], minutes + step))
    return None


def breadth_first(unit, minutes_so_far, depth, order):
    # Shallowest route first; among equally shallow routes, oldest first (a plain queue).
    return (depth, order)


result = search(hospital, START, GOAL, breadth_first)
print("breadth-first")
print("   route   :", " -> ".join(result["route"]))
print("   minutes :", result["minutes"])
print("   stops   :", result["stops"])
print("   expanded:", result["expanded"], "units")

## Part 2: Four more policies

Breadth-first and depth-first policies are provided. They differ by a sign, since depth-first
search takes routes from the opposite end of the same queue. The remaining three policies are
left for you to implement, and each requires one line. As provided, all three return the
breadth-first key, so the cell runs and all five policies give identical results. This is the
main point of the exercise: the five algorithms are a single procedure with different sort keys.

- **Uniform-cost:** order by minutes already spent
- **Greedy best-first:** order by estimated minutes remaining, ignoring cost so far
- **A\*:** order by minutes spent plus estimated minutes remaining

In [ ]:
def estimated_remaining(unit):
    """Straight-line minutes from a unit to the OR. Never an overestimate, because
    no corridor is faster than walking in a straight line."""
    return round(straight_line(unit, GOAL), 1)


def depth_first(unit, minutes_so_far, depth, order):
    # Deepest route first; among equally deep routes, newest first (a plain stack).
    return (-depth, -order)


def uniform_cost(unit, minutes_so_far, depth, order):
    # TODO: order by the cost already spent getting here.
    return (depth, order)


def greedy_best_first(unit, minutes_so_far, depth, order):
    # TODO: order by estimated_remaining(unit) alone -- this is the deck's f(n) = h(n).
    return (depth, order)


def a_star(unit, minutes_so_far, depth, order):
    # TODO: order by f(n) = g(n) + h(n): minutes spent plus minutes estimated to remain.
    return (depth, order)


POLICIES = {
    "breadth-first": breadth_first,
    "depth-first": depth_first,
    "uniform-cost": uniform_cost,
    "greedy best-first": greedy_best_first,
    "A* (straight-line h)": a_star,
}

print("estimated minutes remaining, from each unit to OR 3:")
for unit in sorted(UNITS):
    print(f"   {unit:22s} {estimated_remaining(unit):5.1f}")

### Predictions

Before building the comparison table, agree as a class on answers to the following:

1. Which policies will return the minimum-cost route in minutes?
2. Which policy will expand the **fewest** units?
3. Is the answer to (2) the same as the answer to (1)?

Then complete the three TODOs above, re-run that cell, and run the cell below.

In [ ]:
comparison = []
for name, policy in POLICIES.items():
    found = search(hospital, START, GOAL, policy)
    comparison.append({
        "algorithm": name,
        "minutes": found["minutes"],
        "stops": found["stops"],
        "units expanded": found["expanded"],
        "route": " -> ".join(SHORT_NAMES[unit] for unit in found["route"]),
    })

print(pd.DataFrame(comparison).to_string(index=False))

With the TODOs completed, the table shows three results.

**Breadth-first search optimizes the wrong quantity.** It returns the route with the fewest
stops (three), which passes through the freight elevator and takes about eight minutes longer
than the optimal route. Minimizing the number of edges is equivalent to minimizing cost only
when all edges have equal cost, which holds in a uniform grid but not on this map.

**Greedy best-first search expands the fewest units and returns the worst route.** It expands
four units because it always moves toward the unit geometrically closest to the OR. The south
lobby is 2.2 minutes from the OR in straight-line distance, so greedy search selects it and
must then use the freight elevator. This is a worked example of the *Greedy Search: What can go
wrong?* slide.

**A\* expands more units than greedy search and returns the optimal route.** It finds the same
route as uniform-cost search while expanding fewer units. The cell below lists the units that
uniform-cost search expands and A\* does not.

In [ ]:
ucs_order = search(hospital, START, GOAL, uniform_cost)["expansion_order"]
astar_order = search(hospital, START, GOAL, a_star)["expansion_order"]

print("expanded by uniform-cost but NOT by A*:")
for unit in ucs_order:
    if unit not in astar_order:
        print(f"   {unit:22s} (straight-line estimate to OR: {estimated_remaining(unit):.1f} min)")
print()
print("Uniform-cost search expands these units because they are close to the ED. A* does not,")
print("because their low cost so far is outweighed by their large estimated distance to the OR.")

### Depth-first search

Depth-first search may appear competitive in the table above, but its result depends on the
order in which corridors are listed. The cell below changes only the tie-break, i.e., which
child of the deepest route is removed first. Both versions are valid depth-first search.

In [ ]:
def depth_first_other_child(unit, minutes_so_far, depth, order):
    return (-depth, order)      # same algorithm, opposite tie-break


for label, policy in [("depth-first (newest child)", depth_first),
                      ("depth-first (oldest child)", depth_first_other_child)]:
    found = search(hospital, START, GOAL, policy)
    print(f"{label:30s} {found['minutes']:5.1f} min, {found['expanded']:2d} expanded")
    print(f"{'':30s} {' -> '.join(SHORT_NAMES[unit] for unit in found['route'])}")

## Part 3: Inadmissible heuristics

A* is guaranteed to return an optimal route only when its heuristic is **admissible**, meaning
it never overestimates the true remaining cost. Straight-line distance is admissible on this map
by construction, because every corridor's traversal time is its straight-line length multiplied
by a factor of at least 1.0.

Suppose an analyst reviews a year of transport logs and finds that actual transport times average
about five times the straight-line estimate once elevator waits and doors are included. The
analyst "calibrates" the heuristic by scaling it accordingly, and the scaled estimates are much
closer to observed times on average. Run the cell below and compare the results across scaling
factors.

In [ ]:
# SCALE is read each time the policy runs, so changing it below changes the heuristic.
SCALE = 1


def a_star_calibrated(unit, minutes_so_far, depth, order):
    return (minutes_so_far + SCALE * estimated_remaining(unit), order)


broken = []
for SCALE in [1, 2, 3, 5]:
    found = search(hospital, START, GOAL, a_star_calibrated)
    broken.append({"heuristic": f"h x {SCALE}", "minutes": found["minutes"],
                   "units expanded": found["expanded"],
                   "route": " -> ".join(SHORT_NAMES[unit] for unit in found["route"])})

greedy = search(hospital, START, GOAL, greedy_best_first)
broken.append({"heuristic": "(greedy, for reference)", "minutes": greedy["minutes"],
               "units expanded": greedy["expanded"],
               "route": " -> ".join(SHORT_NAMES[unit] for unit in greedy["route"])})

print(pd.DataFrame(broken).to_string(index=False))

Every row below the first expands fewer units, and every row below the first returns a
suboptimal route. At `h x 5` the search no longer behaves like A*: it expands four units, uses
the freight elevator, and returns the same route as greedy search, eight minutes slower than
optimal for a patient being moved to an operating room. A benchmark based on search speed alone
would favor this version, and nothing in the output indicates that the optimality guarantee no
longer holds.

Deliberately scaling a heuristic is an established technique (weighted A*), and it comes with a
bound: the returned route costs at most `weight` times the optimal cost. The error is not the
scaling itself but reporting the result as the shortest route. To measure the overestimation
directly, the cell below computes the true remaining cost from every unit by running
uniform-cost search from that unit to the OR with the existing `search` function.

In [ ]:
admissibility = []
for unit in sorted(UNITS):
    true_remaining = search(hospital, unit, GOAL, uniform_cost)["minutes"]
    estimate = estimated_remaining(unit)
    admissibility.append({
        "unit": unit,
        "h (straight line)": estimate,
        "h x 5": round(5 * estimate, 1),
        "true remaining": true_remaining,
        "h ok": estimate <= true_remaining,
        "h x 5 ok": 5 * estimate <= true_remaining,
    })

table = pd.DataFrame(admissibility)
print(table.to_string(index=False))
print()
print(f"straight-line heuristic overestimates at {(~table['h ok']).sum()} of {len(table)} units")
print(f"scaled heuristic overestimates at        {(~table['h x 5 ok']).sum()} of {len(table)} units")

A single overestimate can be enough. A* terminates when it removes the goal from the frontier,
and an inflated estimate at a unit on the optimal route makes that route appear more expensive
than it is. The search then commits to a different route and does not return to the optimal one.
Admissibility is therefore not a measure of heuristic quality but a precondition for the
optimality guarantee.

In practice, a heuristic fitted to historical data, such as average observed transport times or
a model of typical delays, is admissible only by coincidence. The guarantee depends on a
defensible lower bound (e.g., travel cannot be faster than a straight line), not on accuracy on
average.

## Discussion

1. Greedy search expanded four units and returned a route eight minutes slower than optimal. For
   a single patient being moved to an OR, eight minutes is significant; for a scheduler planning
   ten thousand transports overnight, the number of expansions is significant. Where is the
   crossover, and who in a health system should decide on that trade-off?
2. The straight-line heuristic is admissible because it is a physical lower bound. What lower
   bound could be justified for a differential-diagnosis search, where the distance to the goal
   is the number and cost of tests needed to reach a confident diagnosis?
3. The map assumes every corridor is always available. In practice, elevators are taken out of
   service, patients on contact precautions require alternate routes, and units stop accepting
   patients at shift change. Do these constraints change the algorithm, the graph, or the
   definition of the goal?

## Solutions

Completed versions of the three TODOs, in a markdown cell so they are not executed.

```python
def uniform_cost(unit, minutes_so_far, depth, order):
    return (minutes_so_far, order)


def greedy_best_first(unit, minutes_so_far, depth, order):
    return (estimated_remaining(unit), order)


def a_star(unit, minutes_so_far, depth, order):
    return (minutes_so_far + estimated_remaining(unit), order)
```

None of the five algorithms has a separate implementation; there is no queue class, stack, or
priority-queue variant. All five use the same loop over the same frontier, and the difference
between an algorithm that guarantees the minimum-cost route and one that does not is confined to
a single `return` statement.

Two details in `search` are easy to overlook:

- `if unit in expanded: continue` makes this graph search rather than tree search. Without it,
  the search revisits units and can loop indefinitely on a map with cycles such as this one.
- The `order` counter in each key provides a deterministic tie-break. Without it, two routes with
  equal priority would be compared on the next element of the tuple, and the results would
  depend on dictionary ordering rather than on the algorithm. Reproducible behavior has to be
  built into the search explicitly.